# LoRA Fine-Tuning LLaMA 3.2 1B verifier
This notebook fine-tunes `meta-llama/Llama-3.2-1B` as a binary reward/verifier model over claim-level reasoning paths.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers peft evaluate tomli

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 13.2 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import time
import random
import datetime
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split


import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score
import evaluate

from peft import LoraConfig, get_peft_model


In [ ]:
torch.set_default_dtype(torch.float32)


In [ ]:
f1_metric = evaluate.load("f1")

def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))


def normalize_label(x):
    """
    Normalize dataset labels and generated verdict labels so that:
      True / SUPPORTS -> true
      False / REFUTES -> false
      Conflicting / CONFLICTING -> conflicting
    """
    x = str(x).strip().lower()

    label_map = {
        "true": "true",
        "supports": "true",
        "support": "true",
        "supported": "true",

        "false": "false",
        "refutes": "false",
        "refute": "false",
        "refuted": "false",

        "conflicting": "conflicting",
        "conflict": "conflicting",
        "not enough info": "conflicting",
        "nei": "conflicting",
    }

    return label_map.get(x, x)


def clean_reasoning_trace(text):
    """
    Remove a final generated label line from a reasoning trace.
    Keeps the reasoning and justification content.
    """
    text = str(text)

    text = re.sub(
        r"(?im)^\s*(#+\s*)?\**\s*\[?\s*Label\s*\]?\s*:?\s*\**\s*"
        r"(SUPPORTS|REFUTES|CONFLICTING|TRUE|FALSE)\s*\**\s*$",
        "",
        text,
    )

    return text.strip()


# Backward-compatible name because the old Llama notebook used remove_label_pattern().
def remove_label_pattern(text):
    return clean_reasoning_trace(text)


def load_json_dataset(path):
    """
    Loads either:
      1. JSON array files: [ {...}, {...} ]
      2. JSONL files: one JSON object per line
    Returns a pandas DataFrame.
    """
    try:
        return pd.read_json(path, lines=False)
    except ValueError:
        return pd.read_json(path, lines=True)


def load_json_records(path):
    """
    Loads validation/test data as a list of dictionaries.
    Handles both JSON array and JSONL.
    """
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read().strip()

    if raw.startswith("["):
        return json.loads(raw)

    return [json.loads(line) for line in raw.splitlines() if line.strip()]


def evidence_to_text(evidences, max_evidences=None, max_chars_per_evidence=None):
    if evidences is None:
        return ""

    if isinstance(evidences, list):
        selected = evidences if max_evidences is None else evidences[:max_evidences]
        cleaned = []

        for i, ev in enumerate(selected):
            ev_text = str(ev).replace("\n", " ").strip()

            if max_chars_per_evidence is not None and len(ev_text) > max_chars_per_evidence:
                ev_text = ev_text[:max_chars_per_evidence].rstrip() + " ..."

            cleaned.append(f"{i + 1}. {ev_text}")

        return "\n".join(cleaned)

    ev_text = str(evidences).replace("\n", " ").strip()

    if max_chars_per_evidence is not None and len(ev_text) > max_chars_per_evidence:
        ev_text = ev_text[:max_chars_per_evidence].rstrip() + " ..."

    return ev_text


def make_input_text(claim, evidences, candidate_verdict, reasoning_trace):
    """
    Build the verifier input.

    Path-specific fields are intentionally placed before evidence. With decoder-only
    Llama and max_length truncation, this prevents long final-test evidence from
    removing Candidate verdict and Reasoning path from the tokenized input.
    """
    evidence_text = evidence_to_text(evidences)
    reasoning_trace = clean_reasoning_trace(reasoning_trace)

    return (
        f"Claim:\n{claim}\n\n"
        f"Candidate verdict:\n{candidate_verdict}\n\n"
        f"Reasoning path:\n{reasoning_trace}\n\n"
        f"Evidence:\n{evidence_text}"
    )


def validate_reasoning_path_requirements(raw_df, expected_paths=20):
    """
    Checks the dataset structure required by the CLEF 2026 verifier setup:
      - each claim row has Reasoning_traces
      - each claim row has Verdict_list
      - train/validation rows can be flattened into one row per reasoning path
      - expected number of reasoning paths is 20 for train/validation claims
      - each row has claim, evidences, and label
    """
    required_columns = [
        "Reasoning_traces",
        "Verdict_list",
        "claim",
        "evidences",
        "label",
    ]

    missing_columns = [col for col in required_columns if col not in raw_df.columns]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    bad_rows = []

    for idx, row in raw_df.reset_index(drop=True).iterrows():
        traces = row["Reasoning_traces"]
        verdicts = row["Verdict_list"]

        trace_count = len(traces) if isinstance(traces, list) else None
        verdict_count = len(verdicts) if isinstance(verdicts, list) else None

        if trace_count != expected_paths or verdict_count != expected_paths:
            bad_rows.append({
                "row": idx,
                "trace_count": trace_count,
                "verdict_count": verdict_count,
            })

    if bad_rows:
        print(
            f"Warning: {len(bad_rows)} rows do not have exactly "
            f"{expected_paths} reasoning traces and {expected_paths} verdicts."
        )
        print("First problematic rows:", bad_rows[:10])
    else:
        print(
            f"Requirement check passed: every claim row has "
            f"{expected_paths} reasoning traces and {expected_paths} verdicts."
        )

    evidence_lengths = raw_df["evidences"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

    print("Evidence count per claim:")
    print(evidence_lengths.describe())

    return bad_rows


def flatten_verifier_dataset(raw_df):
    """
    Converts one claim row with 20 reasoning traces into 20 verifier examples.

    Original row:
      Reasoning_traces = [trace_1, ..., trace_20]
      Verdict_list     = [verdict_1, ..., verdict_20]
      label            = gold label

    New rows:
      input_text = claim + candidate verdict + one reasoning trace + evidence
      Class      = 1 if Verdict_list[i] == label else 0

    The split must happen before calling this function, at original-claim level,
    so that reasoning paths from the same claim cannot leak across train/validation.
    """
    rows = []

    for claim_id, row in raw_df.reset_index(drop=True).iterrows():
        claim = row["claim"]
        evidences = row.get("evidences", [])
        gold_label = normalize_label(row["label"])

        reasoning_traces = row["Reasoning_traces"]
        verdict_list = row["Verdict_list"]

        if not isinstance(reasoning_traces, list):
            continue

        if not isinstance(verdict_list, list):
            continue

        n = min(len(reasoning_traces), len(verdict_list))

        for path_id in range(n):
            candidate_verdict_raw = verdict_list[path_id]
            candidate_verdict = normalize_label(candidate_verdict_raw)
            reasoning_trace = reasoning_traces[path_id]

            input_text = make_input_text(
                claim=claim,
                evidences=evidences,
                candidate_verdict=candidate_verdict_raw,
                reasoning_trace=reasoning_trace,
            )

            rows.append({
                "claim_id": claim_id,
                "path_id": path_id,
                "claim": claim,
                "gold_label": gold_label,
                "candidate_verdict": candidate_verdict,
                "input_text": input_text,
                "Class": int(candidate_verdict == gold_label),
            })

    return pd.DataFrame(rows)


def print_trainable_parameters(model):
    trainable_params, all_param = 0, 0

    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_param} || "
        f"trainable%: {100 * trainable_params / all_param:.2f}"
    )


In [ ]:
class CustomClassifier(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=1,
        hidden_dim=None,
        dropout_value=0.1,
        freeze_base_layer=True,
        use_lora=False,
        is_base_encoder=False,
        lora_rank=8,
        lora_alpha=16,
    ):
        super().__init__()

        self.model = AutoModel.from_pretrained(model_name)
        self.is_base_encoder = is_base_encoder

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

        if use_lora:
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                target_modules=["q_proj", "k_proj", "v_proj"],
                lora_dropout=0.0,
                bias="none",
            )
            self.model = get_peft_model(self.model, lora_config)

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        if self.is_base_encoder and hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            pooled_output = outputs.pooler_output
        else:
            sequence_output = outputs.last_hidden_state

            # Decoder-only models such as Gemma do not have pooler_output.
            # Because we pad to max_length, sequence_output[:, -1, :] may point to padding.
            # So we select the final non-padding token for each sequence.
            last_token_indices = attention_mask.sum(dim=1) - 1
            batch_indices = torch.arange(
                sequence_output.size(0),
                device=sequence_output.device,
            )

            pooled_output = sequence_output[batch_indices, last_token_indices]

        pooled_output = pooled_output.float()
        logits = self.classifier(pooled_output)

        return logits

In [ ]:
class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.tokenizer.padding_side = "right"

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


In [ ]:
class TrainerModule:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        epochs,
        lr,
        patience,
        output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs
        self.patience = patience

        self.optimizer = AdamW(self.model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def train(self):
        best_val_loss = float("inf")
        patience_counter = 0

        for epoch_idx in range(self.epochs):
            epoch_num = epoch_idx + 1

            print(f"\nEpoch {epoch_num}/{self.epochs}")
            self.model.train()

            total_loss = 0.0
            total_acc = 0.0

            train_pbar = tqdm(
                self.train_loader,
                desc=f"Training {epoch_num}/{self.epochs}",
                total=len(self.train_loader),
                unit="batch",
                dynamic_ncols=True,
                leave=True,
            )

            for step, batch in enumerate(train_pbar, start=1):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask).squeeze(1)

                loss = self.loss_fn(logits, labels)

                loss.backward()
                self.optimizer.step()
                self.scheduler.step()

                batch_loss = loss.item()

                preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
                labels_np = labels.long().cpu().numpy()

                batch_acc = accuracy_score(labels_np, preds)

                total_loss += batch_loss
                total_acc += batch_acc

                running_loss = total_loss / step
                running_acc = total_acc / step

                train_pbar.set_postfix({
                    "loss": f"{running_loss:.4f}",
                    "acc": f"{running_acc:.4f}",
                    "lr": f"{self.optimizer.param_groups[0]['lr']:.2e}",
                })

            avg_train_loss = total_loss / len(self.train_loader)
            avg_train_acc = total_acc / len(self.train_loader)

            print(f"Train Loss: {avg_train_loss:.4f}")
            print(f"Train Acc: {avg_train_acc:.4f}")

            val_loss, val_acc = self.evaluate(epoch_num)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0

                tokenizer.save_pretrained(self.output_dir)
                torch.save(
                    self.model.state_dict(),
                    os.path.join(self.output_dir, "best_model"),
                )

                print(f"Saved new best model with Val Loss: {best_val_loss:.4f}")
            else:
                patience_counter += 1
                print(f"No improvement. Patience: {patience_counter}/{self.patience}")

                if patience_counter >= self.patience:
                    print("Early stopping.")
                    break

    def evaluate(self, epoch_num):
        self.model.eval()

        total_loss = 0.0
        total_acc = 0.0

        val_pbar = tqdm(
            self.val_loader,
            desc=f"Validating {epoch_num}/{self.epochs}",
            total=len(self.val_loader),
            unit="batch",
            dynamic_ncols=True,
            leave=True,
        )

        with torch.no_grad():
            for step, batch in enumerate(val_pbar, start=1):
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask).squeeze(1)
                loss = self.loss_fn(logits, labels)

                batch_loss = loss.item()

                preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
                labels_np = labels.long().cpu().numpy()

                batch_acc = accuracy_score(labels_np, preds)

                total_loss += batch_loss
                total_acc += batch_acc

                running_loss = total_loss / step
                running_acc = total_acc / step

                val_pbar.set_postfix({
                    "val_loss": f"{running_loss:.4f}",
                    "val_acc": f"{running_acc:.4f}",
                })

        avg_val_loss = total_loss / len(self.val_loader)
        avg_val_acc = total_acc / len(self.val_loader)

        print(f"Val Loss: {avg_val_loss:.4f}")
        print(f"Val Acc: {avg_val_acc:.4f}")

        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, f"model_epoch_{epoch_num}"),
        )

        return avg_val_loss, avg_val_acc

In [ ]:
class VerifierEvaluator:
    def __init__(
        self,
        model_path,
        tokenizer_path,
        base_model,
        use_decomp=True,
        device="cuda",
        max_length=None,
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.use_decomp = use_decomp
        self.max_length = max_length if max_length is not None else globals().get("MAX_LENGTH", 512)

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.tokenizer.padding_side = "right"

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = CustomClassifier(
            base_model,
            use_lora=True,
            is_base_encoder=False,
        )

        self.model.load_state_dict(
            torch.load(model_path, map_location=self.device)
        )

        self.model.to(self.device)
        self.model.eval()

    def build_input_text(self, claim, evidences, verdict, justification):
        return make_input_text(
            claim=claim,
            evidences=evidences,
            candidate_verdict=verdict,
            reasoning_trace=justification,
        )

    def encode_input(
        self,
        claim,
        evidences,
        verdict,
        justification,
        max_length=None,
    ):
        text = self.build_input_text(
            claim=claim,
            evidences=evidences,
            verdict=verdict,
            justification=justification,
        )

        if max_length is None:
            max_length = self.max_length

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, evidences, verdict, justification):
        ids, mask = self.encode_input(
            claim=claim,
            evidences=evidences,
            verdict=verdict,
            justification=justification,
        )

        with torch.no_grad():
            logit = self.model(ids, mask).squeeze().item()
            probability = torch.sigmoid(torch.tensor(logit)).item()

        return probability


def debug_tokenized_path_variation(evaluator, sample, max_paths=5):
    """
    Sanity check for final-test inference.
    If all tokenized inputs are identical, the verifier cannot rank reasoning paths.
    """
    reasoning_traces = sample["Reasoning_traces"]
    candidate_verdicts = sample["Verdict_list"]
    n = min(max_paths, len(reasoning_traces), len(candidate_verdicts))

    fingerprints = []
    for path_idx in range(n):
        text = evaluator.build_input_text(
            claim=sample["claim"],
            evidences=sample.get("evidences", []),
            verdict=candidate_verdicts[path_idx],
            justification=clean_reasoning_trace(reasoning_traces[path_idx]),
        )
        enc = evaluator.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=evaluator.max_length,
            return_tensors="pt",
        )
        active_ids = enc["input_ids"][0][enc["attention_mask"][0].bool()]
        decoded = evaluator.tokenizer.decode(active_ids, skip_special_tokens=True)
        fingerprints.append(tuple(active_ids.tolist()))

        print(
            f"path {path_idx}: tokens={len(active_ids)}, "
            f"has_candidate_verdict={'Candidate verdict:' in decoded}, "
            f"has_reasoning_path={'Reasoning path:' in decoded}"
        )

    unique_count = len(set(fingerprints))
    print(f"Unique tokenized inputs among first {n} paths: {unique_count}/{n}")

    if unique_count == 1 and n > 1:
        raise RuntimeError(
            "All checked tokenized path inputs are identical. "
            "The verifier is not seeing path-specific content."
        )


In [ ]:
import json
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from huggingface_hub import login
from google.colab import userdata

login(userdata.get("HF_TOK_READ"))

BASE_MODEL = "meta-llama/Llama-3.2-1B"
TEST_JSON = "/content/drive/MyDrive/BioNLP/clef_spanish_test_final.json"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

with open(TEST_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

total_inputs = sum(
    min(len(sample["Reasoning_traces"]), len(sample["Verdict_list"]))
    for sample in data
)

lengths = []

with tqdm(total=total_inputs, desc="Tokenizing verifier inputs", unit="input") as pbar:
    for sample_idx, sample in enumerate(data):
        claim = sample["claim"]
        evidences = sample.get("evidences", [])
        reasoning_traces = sample["Reasoning_traces"]
        verdicts = sample["Verdict_list"]

        for reasoning_trace, verdict in zip(reasoning_traces, verdicts):
            text = make_input_text(
                claim=claim,
                evidences=evidences,
                candidate_verdict=verdict,
                reasoning_trace=reasoning_trace,
            )

            token_count = len(
                tokenizer(
                    text,
                    truncation=False,
                    add_special_tokens=True,
                )["input_ids"]
            )

            lengths.append(token_count)
            pbar.update(1)

lengths = np.array(lengths)

print("Number of verifier inputs:", len(lengths))
print("min:", int(lengths.min()))
print("median:", int(np.percentile(lengths, 50)))
print("p75:", int(np.percentile(lengths, 75)))
print("p90:", int(np.percentile(lengths, 90)))
print("p95:", int(np.percentile(lengths, 95)))
print("p99:", int(np.percentile(lengths, 99)))
print("max:", int(lengths.max()))


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Tokenizing verifier inputs:   0%|          | 0/23280 [00:00<?, ?input/s]

Number of verifier inputs: 23280
min: 121
median: 3107
p75: 3789
p90: 4403
p95: 4648
p99: 5302
max: 6467


In [ ]:
# ===== PATH PLACEHOLDERS =====
TRAIN_CSV = "/content/drive/MyDrive/BioNLP/spanish_train.json"
VAL_CSV   = "/content/drive/MyDrive/BioNLP/spanish_val.json"

BASE_MODEL = "meta-llama/Llama-3.2-1B"

#OUTPUT_DIR = "/content/drive/MyDrive/BioNLP/projectBioNLP/models/" + BASE_MODEL + "_ENG"
OUTPUT_DIR = "/content/sample_data/models/" + BASE_MODEL + "_SPA"

print(f"OUTPUT_DIR: {OUTPUT_DIR}")

from transformers import AutoConfig

MODEL_CONFIG = AutoConfig.from_pretrained(BASE_MODEL)

MODEL_MAX_POSITIONS = getattr(MODEL_CONFIG, "max_position_embeddings", None)
print("Model max_position_embeddings:", MODEL_MAX_POSITIONS)

print("MAX_LENGTH 95th percentile:", int(np.percentile(lengths, 95)))
print("MAX_LENGTH 99th percentile:", int(np.percentile(lengths, 99)))

MAX_LENGTH = 1610
BATCH_SIZE = 4
EPOCHS = 3
LR = 1e-4


OUTPUT_DIR: /content/sample_data/models/meta-llama/Llama-3.2-1B_SPA
Model max_position_embeddings: 131072
MAX_LENGTH 95th percentile: 4648
MAX_LENGTH 99th percentile: 5302


In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get("HF_TOK_READ"))

raw_train_df = load_json_dataset(TRAIN_CSV)
raw_val_df = load_json_dataset(VAL_CSV)

print("Train raw columns:", raw_train_df.columns.tolist())
print("Validation raw columns:", raw_val_df.columns.tolist())

print("Number of original train claim rows:", len(raw_train_df))
print("Number of original validation claim rows:", len(raw_val_df))

required_columns = [
    "Reasoning_traces",
    "Verdict_list",
    "claim",
    "evidences",
    "label",
]

missing_train_columns = [col for col in required_columns if col not in raw_train_df.columns]
missing_val_columns = [col for col in required_columns if col not in raw_val_df.columns]

if missing_train_columns:
    raise ValueError(f"Missing required columns in TRAIN_CSV: {missing_train_columns}")

if missing_val_columns:
    raise ValueError(f"Missing required columns in VAL_CSV: {missing_val_columns}")

raw_train_df["normalized_label"] = raw_train_df["label"].apply(normalize_label)
raw_val_df["normalized_label"] = raw_val_df["label"].apply(normalize_label)

print("\nTrain gold-label distribution:")
print(raw_train_df["normalized_label"].value_counts())

print("\nValidation gold-label distribution:")
print(raw_val_df["normalized_label"].value_counts())

# No train_test_split here.
# Use the full TRAIN_CSV for training and the full VAL_CSV for validation.
train_claims_df = raw_train_df
val_claims_df = raw_val_df

train_df = flatten_verifier_dataset(train_claims_df)
val_df = flatten_verifier_dataset(val_claims_df)

print("\nOriginal train claim rows:", len(train_claims_df))
print("Original validation claim rows:", len(val_claims_df))

print("Train verifier rows:", len(train_df))
print("Validation verifier rows:", len(val_df))

print("\nExpected train verifier rows if every claim has 20 paths:", len(train_claims_df) * 20)
print("Expected validation verifier rows if every claim has 20 paths:", len(val_claims_df) * 20)

print("\nTrain class distribution:")
print(train_df["Class"].value_counts())

print("\nValidation class distribution:")
print(val_df["Class"].value_counts())

print("\nExample flattened train rows:")
print(train_df[["claim_id", "path_id", "gold_label", "candidate_verdict", "Class"]].head(30))

print("\nExample flattened validation rows:")
print(val_df[["claim_id", "path_id", "gold_label", "candidate_verdict", "Class"]].head(30))

print("\nExample model input:")
print(train_df["input_text"].iloc[0][:2000])

if train_df.empty:
    raise ValueError("train_df is empty after flattening. Check Reasoning_traces and Verdict_list in TRAIN_CSV.")

if val_df.empty:
    raise ValueError("val_df is empty after flattening. Check Reasoning_traces and Verdict_list in VAL_CSV.")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "right"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = TextDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Train dataset rows: {len(train_dataset)}")
print(f"Validation dataset rows: {len(val_dataset)}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Validation batches per epoch: {len(val_loader)}")
print(f"Batch size: {BATCH_SIZE}")

model = CustomClassifier(
    BASE_MODEL,
    use_lora=True,
    is_base_encoder=False,
)

print_trainable_parameters(model)

trainer = TrainerModule(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LR,
    patience=1,
    output_dir=OUTPUT_DIR,
)

trainer.train()

Train raw columns: ['Reasoning_traces', 'Verdict_list', 'claim', 'evidences', 'label', 'verdict']
Validation raw columns: ['Reasoning_traces', 'Verdict_list', 'claim', 'evidences', 'label', 'verdict']
Number of original train claim rows: 2246
Number of original validation claim rows: 562

Train gold-label distribution:
normalized_label
false          1888
conflicting     206
true            152
Name: count, dtype: int64

Validation gold-label distribution:
normalized_label
false          473
conflicting     51
true            38
Name: count, dtype: int64

Original train claim rows: 2246
Original validation claim rows: 562
Train verifier rows: 44920
Validation verifier rows: 11240

Expected train verifier rows if every claim has 20 paths: 44920
Expected validation verifier rows if every claim has 20 paths: 11240

Train class distribution:
Class
1    29207
0    15713
Name: count, dtype: int64

Validation class distribution:
Class
1    7131
0    4109
Name: count, dtype: int64

Example fla

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

trainable params: 1181697 || all params: 1236996097 || trainable%: 0.10

Epoch 1/3


Training 1/3:   0%|          | 0/11230 [00:00<?, ?batch/s]

...

Val Loss: 0.5441
Val Acc: 0.8743
Saved new best model with Val Loss: 0.5441

Epoch 2/3


Training 2/3:   0%|          | 0/11230 [00:00<?, ?batch/s]

Train Loss: 0.0124
Train Acc: 0.9964


Validating 2/3:   0%|          | 0/2810 [00:00<?, ?batch/s]

Val Loss: 0.6391
Val Acc: 0.8738
No improvement. Patience: 1/1
Early stopping.


In [ ]:
from pathlib import Path
import shutil

OUTPUT_DIR = Path("/content/sample_data/models/" + BASE_MODEL + "_SPA")
target_dir = Path("/content/drive/MyDrive/BioNLP/models/" + BASE_MODEL + "_SPA")

# Ensure target directory exists
target_dir.mkdir(parents=True, exist_ok=True)

# Copy best_model file
source_best_model = OUTPUT_DIR / "best_model"
target_best_model = target_dir / "best_model"

if not source_best_model.exists():
    raise FileNotFoundError(f"Missing file: {source_best_model}")

if not source_best_model.is_file():
    raise ValueError(f"Expected best_model to be a file, but found a directory: {source_best_model}")

shutil.copy2(source_best_model, target_best_model)

print(f"Copied file:")
print(f"  {source_best_model}")
print(f"  -> {target_best_model}")

# Copy every .json file from the source directory
json_files = list(OUTPUT_DIR.glob("*.json"))

if not json_files:
    print(f"No .json files found in {OUTPUT_DIR}")
else:
    for json_file in json_files:
        target_json_file = target_dir / json_file.name
        shutil.copy2(json_file, target_json_file)
        print(f"Copied JSON:")
        print(f"  {json_file}")
        print(f"  -> {target_json_file}")

Copied file:
  /content/sample_data/models/meta-llama/Llama-3.2-1B_SPA/best_model
  -> /content/drive/MyDrive/BioNLP/models/meta-llama/Llama-3.2-1B_SPA/best_model
Copied JSON:
  /content/sample_data/models/meta-llama/Llama-3.2-1B_SPA/tokenizer.json
  -> /content/drive/MyDrive/BioNLP/models/meta-llama/Llama-3.2-1B_SPA/tokenizer.json
Copied JSON:
  /content/sample_data/models/meta-llama/Llama-3.2-1B_SPA/tokenizer_config.json
  -> /content/drive/MyDrive/BioNLP/models/meta-llama/Llama-3.2-1B_SPA/tokenizer_config.json


In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get("HF_TOK_READ"))

OUTPUT_DIR = "/content/drive/MyDrive/BioNLP/models/" + BASE_MODEL + "_SPA"

test_data = load_json_records(VAL_CSV)

predictions_FULL = []
predictions_COMPACT = []

evaluator = VerifierEvaluator(
    model_path=f"{OUTPUT_DIR}/best_model",
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    use_decomp=True,
    max_length=MAX_LENGTH,
)

for idx, sample in enumerate(tqdm(test_data, desc="Scoring validation samples", unit="claim")):
    verdict_list = []
    verifier_score_list = []
    justification_list = []

    reasoning_traces = sample["Reasoning_traces"]
    candidate_verdicts = sample["Verdict_list"]

    n = min(len(reasoning_traces), len(candidate_verdicts))

    for path_idx in range(n):
        justification = clean_reasoning_trace(reasoning_traces[path_idx])
        candidate_verdict = candidate_verdicts[path_idx]

        score = evaluator.score(
            claim=sample["claim"],
            evidences=sample.get("evidences", []),
            verdict=candidate_verdict,
            justification=justification,
        )

        verdict_list.append(candidate_verdict)
        justification_list.append(justification)
        verifier_score_list.append(score)

    if len(verifier_score_list) == 0:
        best_verdict = None
        best_score = None
        best_reasoning_trace = None
    else:
        best_idx = int(np.argmax(np.array(verifier_score_list)))
        best_verdict = verdict_list[best_idx]
        best_score = verifier_score_list[best_idx]
        best_reasoning_trace = justification_list[best_idx]

    predictions_FULL.append({
        "query_id": idx,
        "Claim": sample["claim"],
        "Label": sample.get("label", None),
        "Verdict_BoN": best_verdict,
        "Best_score": best_score,
        "Best_reasoning_trace": best_reasoning_trace,
        "BoN_Verdict_list": verdict_list,
        "Reasoning_traces": justification_list,
        "score_list": verifier_score_list,
    })

    predictions_COMPACT.append({
        "query_id": idx,
        "Claim": sample["claim"],
        #"Label": sample.get("label", None),
        "Verdict_BoN": best_verdict,
        #"Best_score": best_score,
        #"Best_reasoning_trace": best_reasoning_trace,
        "BoN_Verdict_list": verdict_list,
        "Reasoning_traces": justification_list,
        "score_list": verifier_score_list,
    })

print(f"Generated predictions for {len(predictions_FULL)} samples.")
print("Number of full prediction rows:", len(predictions_FULL))
print("Number of compact prediction rows:", len(predictions_COMPACT))


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Scoring validation samples:   0%|          | 0/562 [00:00<?, ?claim/s]

Generated predictions for 562 samples.
Number of full prediction rows: 562
Number of compact prediction rows: 562


In [ ]:
with open(f"/content/drive/MyDrive/BioNLP/predictions/clef_predictions_validate_Llama_3_2_1B_FULL.json", "w") as fp:
    json.dump(predictions_FULL, fp, indent=4)

with open(f"/content/drive/MyDrive/BioNLP/predictions/clef_predictions_validate_Llama_3_2_1B.json", "w") as fp:
    json.dump(predictions_COMPACT, fp, indent=4)


In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get("HF_TOK_READ"))

OUTPUT_DIR = "/content/drive/MyDrive/BioNLP/models/" + BASE_MODEL + "_SPA"

# ===== FINAL TEST INFERENCE CELL =====

FINAL_TEST_JSON = "/content/drive/MyDrive/BioNLP/clef_spanish_test_final.json"

# Recompute MODEL_TAG here in case the cell is run independently.
MODEL_TAG = BASE_MODEL.replace("/", "_")

final_test_data = load_json_records(FINAL_TEST_JSON)

print(f"Loaded final test samples: {len(final_test_data)}")
print("First sample keys:", final_test_data[0].keys())

evaluator = VerifierEvaluator(
    model_path=f"{OUTPUT_DIR}/best_model",
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    use_decomp=True,
    max_length=MAX_LENGTH,
)

# Sanity check: final-test tokenized inputs must differ across reasoning paths.
# If this fails, the model cannot select a best reasoning path.
debug_tokenized_path_variation(evaluator, final_test_data[0], max_paths=5)

final_predictions_full = []
final_predictions_compact = []

for idx, sample in enumerate(tqdm(final_test_data, desc="Scoring final test samples", unit="claim")):
    query_id = sample.get("query_id", idx)
    claim = sample["claim"]
    evidences = sample.get("evidences", [])

    reasoning_traces = sample["Reasoning_traces"]
    candidate_verdicts = sample["Verdict_list"]

    n = min(len(reasoning_traces), len(candidate_verdicts))

    verdict_list = []
    justification_list = []
    verifier_score_list = []

    for path_idx in range(n):
        candidate_verdict = candidate_verdicts[path_idx]
        justification = clean_reasoning_trace(reasoning_traces[path_idx])

        score = evaluator.score(
            claim=claim,
            evidences=evidences,
            verdict=candidate_verdict,
            justification=justification,
        )

        verdict_list.append(candidate_verdict)
        justification_list.append(justification)
        verifier_score_list.append(float(score))

    if len(verifier_score_list) == 0:
        best_idx = None
        best_verdict = None
        best_score = None
        best_reasoning_trace = None
    else:
        best_idx = int(np.argmax(np.array(verifier_score_list)))
        best_verdict = verdict_list[best_idx]
        best_score = verifier_score_list[best_idx]
        best_reasoning_trace = justification_list[best_idx]

    final_predictions_full.append({
        "query_id": query_id,
        "Claim": claim,
        "Verdict_BoN": best_verdict,
        "Best_path_id": best_idx,
        "Best_score": best_score,
        "Best_reasoning_trace": best_reasoning_trace,
        "BoN_Verdict_list": verdict_list,
        "Reasoning_traces": justification_list,
        "score_list": verifier_score_list,
    })

    final_predictions_compact.append({
        "query_id": idx,
        "Claim": sample["claim"],
        #"Label": sample.get("label", None),
        "Verdict_BoN": best_verdict,
        #"Best_score": best_score,
        #"Best_reasoning_trace": best_reasoning_trace,
        "BoN_Verdict_list": verdict_list,
        "Reasoning_traces": justification_list,
        "score_list": verifier_score_list,
    })

print(f"Generated final predictions: {len(final_predictions_full)}")

#print("\nFull prediction:")
#print(json.dumps(final_predictions_full[0], indent=4, ensure_ascii=False))

#print("\nCompact prediction:")
#print(json.dumps(final_predictions_compact[0], indent=4, ensure_ascii=False))


# ===== SAVE OUTPUTS =====

prediction_dir_drive = "/content/drive/MyDrive/BioNLP/predictions"
prediction_dir_local = f"/content/sample_data/models/{MODEL_TAG}_SPA"

os.makedirs(prediction_dir_drive, exist_ok=True)
os.makedirs(prediction_dir_local, exist_ok=True)

full_drive_path = f"{prediction_dir_drive}/clef_2026_final_english_test_predictions_FULL_{MODEL_TAG}.json"
compact_drive_path = f"{prediction_dir_drive}/clef_2026_final_english_test_predictions_COMPACT_{MODEL_TAG}.json"

full_local_path = f"{prediction_dir_local}/clef_2026_final_english_test_predictions_FULL_{MODEL_TAG}.json"
compact_local_path = f"{prediction_dir_local}/clef_2026_final_english_test_predictions_COMPACT_{MODEL_TAG}.json"

with open(full_drive_path, "w", encoding="utf-8") as fp:
    json.dump(final_predictions_full, fp, indent=4, ensure_ascii=False)

with open(compact_drive_path, "w", encoding="utf-8") as fp:
    json.dump(final_predictions_compact, fp, indent=4, ensure_ascii=False)

with open(full_local_path, "w", encoding="utf-8") as fp:
    json.dump(final_predictions_full, fp, indent=4, ensure_ascii=False)

with open(compact_local_path, "w", encoding="utf-8") as fp:
    json.dump(final_predictions_compact, fp, indent=4, ensure_ascii=False)

print("\nSaved final prediction files:")
print(full_drive_path)
print(compact_drive_path)
print(full_local_path)
print(compact_local_path)


Loaded final test samples: 1164
First sample keys: dict_keys(['Reasoning_traces', 'Verdict_list', 'claim', 'evidences', 'query_index'])


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

path 0: tokens=1610, has_candidate_verdict=True, has_reasoning_path=True
path 1: tokens=1610, has_candidate_verdict=True, has_reasoning_path=True
path 2: tokens=1610, has_candidate_verdict=True, has_reasoning_path=True
path 3: tokens=1610, has_candidate_verdict=True, has_reasoning_path=True
path 4: tokens=1610, has_candidate_verdict=True, has_reasoning_path=True
Unique tokenized inputs among first 5 paths: 5/5


Scoring final test samples:   0%|          | 0/1164 [00:00<?, ?claim/s]

Generated final predictions: 1164

Saved final prediction files:
/content/drive/MyDrive/BioNLP/projectBioNLP/Data/English/predictions/clef_2026_final_english_test_predictions_FULL_meta-llama_Llama-3.2-1B.json
/content/drive/MyDrive/BioNLP/projectBioNLP/Data/English/predictions/clef_2026_final_english_test_predictions_COMPACT_meta-llama_Llama-3.2-1B.json
/content/sample_data/predictions/meta-llama_Llama-3.2-1B_ENG/clef_2026_final_english_test_predictions_FULL_meta-llama_Llama-3.2-1B.json
/content/sample_data/predictions/meta-llama_Llama-3.2-1B_ENG/clef_2026_final_english_test_predictions_COMPACT_meta-llama_Llama-3.2-1B.json
